# Paper-to-Code 1 – ResNet (2015)
**Paper:** He, Zhang, Ren, Sun – *Deep Residual Learning for Image Recognition* – [arXiv:1512.03385](https://arxiv.org/abs/1512.03385) · notes: [[CNN Architectures]] · plan: [[Paper-to-Code Months]]

### The claim we reproduce
> Making a plain network deeper can make its **training** error *worse* – the **degradation problem** (paper Fig. 1). This is not overfitting (it's training error) and, with BatchNorm, not simply vanishing gradients. Adding identity **shortcut connections** – learning a residual $F(x)$ so that a block outputs $F(x) + x$ – removes the problem.

### Our scaled-down setup
The paper's own CIFAR-10 architecture (Sec. 4.2): a 3×3 conv, then 3 stages of $n$ basic blocks with 16/32/64 channels, global average pool, linear layer → $6n+2$ weight layers. We compare **8 vs 32 layers** ($n=1, 5$), plain vs residual, on **5,000 training images for 5 epochs** (≈ 2–3 min on a CPU). The paper used 50k images, 64k iterations and depths 20–110.

**How this works:** cells marked **TODO** are yours (each one is a small function). Run the ✅ check under it.
Experiment cells stay skipped until the checks they need pass. Stuck? Open the hidden ▶ *Solution*.
Everything runs on a laptop CPU – the paper's full-scale setup is listed under *Scale it up*.

In [ ]:
import numpy as np, matplotlib.pyplot as plt, time, torch, torch.nn as nn, torch.nn.functional as F, torchvision
torch.manual_seed(0); torch.set_num_threads(4)
DONE = set()
def check(name, test, hint):
    """Runs test(); ✅ if it returns True. Passed checks unlock the experiment cells further down."""
    try:
        ok = bool(test())
    except NotImplementedError:
        print(f"❌ {name}: not implemented yet – fill in the TODO above."); return
    except Exception as e:
        print(f"❌ {name}: {type(e).__name__}: {e}"); return
    if ok: DONE.add(name); print(f"✅ {name}")
    else: print(f"❌ {name}: {hint}")

def ready(*names):
    missing = [n for n in names if n not in DONE]
    if missing: print("⏭  Skipped – pass these checks first:", ", ".join(missing))
    return not missing

# CIFAR-10 (170 MB, downloaded once into ./data). We only use a 5,000-image training subset.
ds = torchvision.datasets.CIFAR10("data", train=True, download=True)
N_TRAIN = 5000
X = torch.tensor(ds.data[:N_TRAIN]).permute(0, 3, 1, 2).float() / 255
y = torch.tensor(ds.targets[:N_TRAIN])
X = (X - X.mean((0, 2, 3), keepdim=True)) / X.std((0, 2, 3), keepdim=True)   # per-channel standardisation
print(X.shape, y.bincount().tolist())

## 1. The shortcut
When a block keeps the shape, the shortcut is the **identity**. When it halves the spatial size and doubles the channels (first block of stages 2 and 3), the shapes no longer match. The paper offers option **A** (identity + zero-padding of extra channels, used for CIFAR) and option **B** (a 1×1 conv with stride 2 – a *projection*). Implement option B: `nn.Conv2d(cin, cout, 1, stride, bias=False)` followed by `nn.BatchNorm2d(cout)`.

In [ ]:
def make_shortcut(cin, cout, stride):
    # TODO 1: return nn.Identity() if the shape is unchanged, otherwise a 1x1-conv + BatchNorm projection
    raise NotImplementedError  # TODO 1

In [ ]:
def _t_short():
    a, b = make_shortcut(16, 16, 1), make_shortcut(16, 32, 2)
    return isinstance(a, nn.Identity) and b(torch.randn(2, 16, 32, 32)).shape == (2, 32, 16, 16)
check("shortcut", _t_short, "Identity for (16,16,1); for (16,32,2) the output must be (2, 32, 16, 16).")

## 2. The basic block
Paper Fig. 2: conv3×3 → BN → ReLU → conv3×3 → BN, **add the shortcut**, then ReLU. The `residual=False` version is the *plain* baseline – identical layers, no addition.

In [ ]:
class BasicBlock(nn.Module):
    def __init__(self, cin, cout, stride=1, residual=True):
        super().__init__()
        self.residual = residual
        self.conv1 = nn.Conv2d(cin, cout, 3, stride, 1, bias=False); self.bn1 = nn.BatchNorm2d(cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, 1, 1, bias=False);     self.bn2 = nn.BatchNorm2d(cout)
        self.shortcut = make_shortcut(cin, cout, stride)

    def forward(self, x):
        # TODO 2: conv1 → bn1 → relu → conv2 → bn2, (+ shortcut(x) if residual), relu
        raise NotImplementedError  # TODO 2

**Why residuals make depth easy:** if the last BN's scale and shift are zero, $F(x)=0$ and a residual block becomes $\text{ReLU}(x)$ – (almost) the identity, so extra blocks can't hurt. A plain block in the same state outputs **zeros** and destroys the signal. The check tests exactly that.

In [ ]:
def _t_block():
    x = torch.randn(4, 16, 8, 8)
    rb, pb = BasicBlock(16, 16, residual=True), BasicBlock(16, 16, residual=False)
    for b in (rb, pb):
        nn.init.zeros_(b.bn2.weight); nn.init.zeros_(b.bn2.bias)
    down = BasicBlock(16, 32, stride=2)(x)
    return torch.allclose(rb(x), F.relu(x)) and pb(x).abs().max() == 0 and down.shape == (4, 32, 4, 4)
check("block", _t_block, "Residual block with zeroed bn2 must return relu(x); plain block must return 0; check the order of operations.")

## 3. The network ($6n+2$ layers)

In [ ]:
class CifarResNet(nn.Module):
    def __init__(self, n, residual=True, width=16, n_classes=10):
        super().__init__()
        self.stem = nn.Sequential(nn.Conv2d(3, width, 3, 1, 1, bias=False), nn.BatchNorm2d(width), nn.ReLU())
        blocks, cin = [], width
        for stage, cout in enumerate([width, 2 * width, 4 * width]):
            for j in range(n):
                stride = 2 if (j == 0 and stage > 0) else 1
                blocks.append(BasicBlock(cin, cout, stride, residual)); cin = cout
        self.blocks = nn.Sequential(*blocks)
        self.fc = nn.Linear(cin, n_classes)

    def forward(self, x):
        return self.fc(self.blocks(self.stem(x)).mean((2, 3)))   # global average pooling

def n_weight_layers(model):   # 3x3 convs + the final linear layer (1x1 projection convs are not counted, as in the paper)
    return sum(isinstance(m, nn.Conv2d) and m.kernel_size == (3, 3) for m in model.modules()) + 1

if ready("shortcut", "block"):
    for n in (1, 3, 5, 9):
        m = CifarResNet(n)
        print(f"n={n}: {n_weight_layers(m)} layers, {sum(p.numel() for p in m.parameters())/1e3:.0f}k parameters")

## 4. Experiment A – gradients at initialisation
Back-propagate one random batch through fresh networks and measure the gradient norm at the very **first** conv layer. Compare depth 8 with depth 56 ($n=9$).

In [ ]:
grad_ratio = {}
if ready("shortcut", "block"):
    xb, yb = torch.randn(64, 3, 32, 32), torch.randint(0, 10, (64,))
    for residual in (False, True):
        g = {}
        for n in (1, 9):
            torch.manual_seed(0); m = CifarResNet(n, residual)
            F.cross_entropy(m(xb), yb).backward()
            g[n] = m.stem[0].weight.grad.norm().item()
        grad_ratio[residual] = g[9] / g[1]
        print(f"{'residual' if residual else 'plain   '}: first-layer grad norm  8 layers {g[1]:.3g}   56 layers {g[9]:.3g}   ratio {g[9]/g[1]:.3g}")

**Surprise?** With BatchNorm the deep plain net's gradients do not *vanish* – they **explode** at initialisation (a known BN effect, see [Yang et al. 2019](https://arxiv.org/abs/1902.08129)). The residual net stays tame. Either way the plain net is hard to optimise, which is the paper's point.

In [ ]:
if ready("shortcut", "block"):
    check("gradient experiment", lambda: grad_ratio[False] > 10 * grad_ratio[True],
          "Expected the plain 56-layer net's gradient to blow up far more than the residual one's.")

## 5. Experiment B – the degradation problem
Train four nets with the paper's optimiser (SGD, lr 0.1, momentum 0.9, weight decay 1e-4, batch 128) and record **training** loss per epoch.

In [ ]:
EPOCHS = 5
history = {}

def train(n, residual, epochs=EPOCHS):
    torch.manual_seed(0)
    model = CifarResNet(n, residual)
    opt = torch.optim.SGD(model.parameters(), lr=0.1, momentum=0.9, weight_decay=1e-4)
    losses = []
    for ep in range(epochs):
        perm, total = torch.randperm(N_TRAIN), 0.0
        for b in range(0, N_TRAIN, 128):
            idx = perm[b:b + 128]
            loss = F.cross_entropy(model(X[idx]), y[idx])
            opt.zero_grad(); loss.backward(); opt.step()
            total += loss.item() * len(idx)
        losses.append(total / N_TRAIN)
    return losses

if ready("shortcut", "block"):
    for n in (1, 5):
        for residual in (False, True):
            t = time.time(); history[(n, residual)] = train(n, residual)
            print(f"{6*n+2:>2} layers {'residual' if residual else 'plain   '}: train loss per epoch {np.round(history[(n, residual)], 3)}  ({time.time()-t:.0f}s)")
    for (n, residual), h in history.items():
        plt.plot(range(1, EPOCHS + 1), h, ("-" if residual else "--"), label=f"{6*n+2}-layer {'ResNet' if residual else 'plain'}")
    plt.xlabel("epoch"); plt.ylabel("training loss"); plt.legend(); plt.title("Degradation problem (cf. paper Fig. 1/6)"); plt.show()

In [ ]:
if ready("shortcut", "block"):
    h = {k: v[-1] for k, v in history.items()}
    check("degradation reproduced", lambda: h[(5, False)] > h[(1, False)] + 0.1,
          "The 32-layer plain net should end with clearly higher TRAINING loss than the 8-layer plain net.")
    check("residuals fix it", lambda: h[(5, True)] < h[(5, False)] - 0.05,
          "The 32-layer ResNet should train clearly better than the 32-layer plain net.")

<details><summary>▶ Solution</summary>

```python
def make_shortcut(cin, cout, stride):
    if stride == 1 and cin == cout:
        return nn.Identity()
    return nn.Sequential(nn.Conv2d(cin, cout, 1, stride, bias=False), nn.BatchNorm2d(cout))
```

```python
    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.residual:
            out = out + self.shortcut(x)
        return F.relu(out)
```
</details>

## Scale it up (stretch goals)
- Full 50k images, data augmentation (4-pixel pad + random crop + flip), lr 0.1 → divide by 10 at 32k and 48k iterations, 64k iterations (needs a GPU; ~1 h for ResNet-20).
- Paper's Table 6 targets: ResNet-20 8.75 % test error, ResNet-56 6.97 %, ResNet-110 6.43 %.
- Implement option **A** shortcuts (stride-2 subsample + zero-pad channels) and compare with option B.
- Look inside: plot the std of each block's residual output $F(x)$ – the paper (Fig. 7) finds deeper nets have *smaller* responses.

---
## 📝 Write-up (fill in – half a page is enough)
Copy this into your results log in [[Paper-to-Code Months]] when done.

- **Paper and its claim in one sentence:**
- **What I implemented / what I scaled down (and why):**
- **Result:**

  | setting | metric | paper | me |
  |---|---|---|---|
  |  |  |  |  |

- **Did the claim hold at small scale? Why / why not?**
- **One detail the paper glossed over that I had to figure out:**
- **What I would try next with a GPU:**
